# RAG vs No RAG vs Long context on SEC 10-Q filings (2015 → today)

Question: **can a local LLM give the right quarterly revenue and net income of a company?**

| Mode | What the model sees | Which quarters |
|---|---|---|
| **A. No RAG** | Only the question | **Every** quarter since 2015 (cheap, ~2 s each) |
| **B. RAG** | The 6 most relevant passages of the filing | One quarter per year (sample) |
| **C. Long context** | The filing pasted in the prompt (as much as fits) | One quarter per year (sample) |
| **Ground truth** | SEC XBRL data, no AI | Every quarter |

**Main idea:** without RAG, the model can only answer from what it learned during training.
So its accuracy should drop to zero after its training cutoff, while RAG and long context stay high.

The notebook has two parts:
- **Part 1 (sections 4 to 11): revenue and net income.** We know the right answer, so we can measure exactly
  which method reads a filing correctly.
- **Part 2 (sections 12 to 14): General questions for a trader.** Outlook, reasons for a change, risks...
  This information is only in the text of the filing, so this is where AI is actually useful.

## 1. Install Ollama and the Python libraries

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!sudo apt-get -qq update > /dev/null && sudo apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install ollama edgartools rank_bm25 plotly pandas beautifulsoup4 transformers

name, memory.total [MiB]
Tesla T4, 15360 MiB
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>

## 2. Settings
- All amounts are asked **in millions of USD**, so there is no unit confusion.

In [2]:
SEC_EMAIL = "matthieu.moreau@mail.utoronto.ca"
TICKERS = ["AAPL", "MSFT", "NVDA"]
START_YEAR = 2015

# model used for RAG and long context
MODEL = "qwen3:8b"

# models tested without RAG, e.g. ["qwen3:8b", "llama3.1:8b"]
NO_RAG_MODELS = ["qwen3:8b"]

# Embedding model (RAG) and tokenizer to count tokens
EMBED_MODEL = "nomic-embed-text"
TOKENIZER = "Qwen/Qwen3-8B"

# context window (tokens) for each mode
NO_RAG_CONTEXT = 4096
RAG_CONTEXT = 16384
LONG_CONTEXT = 32768

# number of passages given to the model in RAG mode
RAG_TOP_K = 6

# True: delete old results and start again. False: continue an interrupted run
START_FRESH = True

# within 0.5%  -> "exact" | within 10%   -> "close"
EXACT_TOLERANCE = 0.005
CLOSE_TOLERANCE = 0.10

## 3. Start Ollama and download the models

In [3]:
import os
import subprocess
import time
import requests

def start_ollama():
    settings = os.environ.copy()

    # uses less memory with long prompts
    settings["OLLAMA_FLASH_ATTENTION"] = "1"

    # compressed memory for the context
    settings["OLLAMA_KV_CACHE_TYPE"] = "q8_0"

    # keep the model loaded between questions
    settings["OLLAMA_KEEP_ALIVE"] = "60m"

    log_file = open("ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], env=settings, stdout=log_file, stderr=subprocess.STDOUT)

    for second in range(60):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            print("Ollama is running")
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama did not start. Look at ollama.log")

start_ollama()

Ollama is running


In [4]:
all_models = set([MODEL, EMBED_MODEL] + NO_RAG_MODELS)
for model_name in all_models:
    print("Downloading", model_name)
    subprocess.run(["ollama", "pull", model_name], capture_output=True)
!ollama list

NAME                       ID              SIZE      MODIFIED               
qwen3:8b                   500a1f067a9f    5.2 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    About a minute ago        


## 4. Ground truth: every quarter since 2015, from SEC XBRL data
One request per company gives all its reported numbers.
We keep the **3-month** values reported in **10-Q** filings (Q1, Q2, Q3; there is no 10-Q for Q4).

In [5]:
import pandas as pd
from datetime import date
from edgar import Company, set_identity

set_identity(SEC_EMAIL)
SEC_HEADERS = {"User-Agent": f"University RAG project {SEC_EMAIL}"}

# Companies changed the name of their revenue line over the years (new accounting rule in 2018)
REVENUE_NAMES = ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues", "SalesRevenueNet"]
NET_INCOME_NAMES = ["NetIncomeLoss"]


def download_company_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
    response = requests.get(url, headers=SEC_HEADERS, timeout=60)
    response.raise_for_status()
    return response.json()["facts"]["us-gaap"]


def number_of_days(start, end):
    return (date.fromisoformat(end) - date.fromisoformat(start)).days


def get_quarterly_values(company_facts, names):
    """Returns {quarter_end_date: {"value": ..., "accession": ..., "filed": ...}}"""
    quarters = {}
    for name in names:
        if name not in company_facts:
            continue
        for fact in company_facts[name]["units"].get("USD", []):
            if fact.get("form") != "10-Q" or "start" not in fact:
                continue
            days = number_of_days(fact["start"], fact["end"])
            if days < 80 or days > 100:      # skip "6 months" and "9 months" values
                continue
            end = fact["end"]
            if int(end[:4]) < START_YEAR:
                continue
            # A quarter is repeated one year later as a comparison, sometimes under a new name.
            # Keep the FIRST filing, whatever name it used, so we test the right document.
            if end not in quarters or fact["filed"] < quarters[end]["filed"]:
                quarters[end] = {"value": float(fact["val"]), "accession": fact["accn"],
                                 "filed": fact["filed"]}
    return quarters


rows = []
for ticker in TICKERS:
    company = Company(ticker)
    facts = download_company_facts(int(company.cik))
    revenue_by_quarter = get_quarterly_values(facts, REVENUE_NAMES)
    net_income_by_quarter = get_quarterly_values(facts, NET_INCOME_NAMES)

    for quarter_end, revenue in revenue_by_quarter.items():
        net_income = net_income_by_quarter.get(quarter_end)
        rows.append({
            "ticker": ticker,
            "company": company.name,
            "cik": int(company.cik),
            "period": quarter_end,
            "year": int(quarter_end[:4]),
            "accession": revenue["accession"],
            "true_revenue": revenue["value"],
            "true_net_income": net_income["value"] if net_income else None,
        })
    time.sleep(0.2)   # be polite with the SEC servers

quarters = pd.DataFrame(rows).sort_values(["ticker", "period"]).reset_index(drop=True)
print(len(quarters), "quarters found")
quarters.groupby("ticker").period.agg(["count", "min", "max"])

104 quarters found


,count,min,max
ticker,,,
AAPL,35,2015-03-28,2026-06-27
MSFT,34,2015-03-31,2026-03-31
NVDA,35,2015-04-26,2026-07-26


## 5. Choose the sample for RAG and long context, then download those filings
One quarter per company per year (the first 10-Q of each year). This keeps the run under ~1 hour.
Downloading and cleaning the ~36 filings takes about 5 to 10 minutes.

**Why we clean the HTML ourselves:** since 2019, filings are in *inline XBRL*. The HTML hides a big block of
machine data (dates, `...Axis`, `...Member`) that is invisible on screen. If we keep it, it fills the context
window with junk and pushes the real financial statements out.

In [6]:
import re
from bs4 import BeautifulSoup


def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")

    # 1. clean a bit
    for tag in soup.find_all(["ix:header", "script", "style"]):
        tag.decompose()
    for tag in soup.find_all(style=re.compile(r"display:\s*none", re.IGNORECASE)):
        tag.decompose()

    # 2. turn every table into markdown lines: | Net sales | $ | 58,010 |
    for table in soup.find_all("table"):
        lines = []
        for table_row in table.find_all("tr"):
            cells = [cell.get_text(" ", strip=True) for cell in table_row.find_all(["td", "th"])]
            cells = [cell for cell in cells if cell]
            if cells:
                lines.append("| " + " | ".join(cells) + " |")
        table.replace_with("\n\n" + "\n".join(lines) + "\n\n")

    # 3. keep paragraphs separated by a blank line
    for block in soup.find_all(["p", "div", "li", "h1", "h2", "h3", "h4", "br"]):
        block.insert_after("\n\n")

    text = soup.get_text(" ")
    text = re.sub(r"<[^>]{0,1000}>", " ", text)       # any HTML tag that is still there
    text = re.sub(r"[ \t\xa0]+", " ", text)          # many spaces -> one space
    text = re.sub(r"\n\s*\n\s*", "\n\n", text)       # many blank lines -> one
    return text.strip()


def is_xbrl_word(word):
    # words that are machine data or HTML code, not text written
    word = word.strip("|,;()[]'\"")
    if re.search(r"\b(us-gaap|dei|ix|xbrli|srt|iso4217|ixt):|contextRef|unitRef|^<|>$|=\"", word):
        return True                                                           # <ix:nonFraction, name="us-gaap:..."
    if re.fullmatch(r"[a-z][a-z\-]*:[A-Za-z]\w*", word):                      # aapl:Something
        return True
    if re.fullmatch(r"[A-Z]\w*(Member|Axis|Domain|LineItems|Abstract)", word): # CustomerOneMember, SegmentsAxis
        return True
    if re.fullmatch(r"P\d+[YMD]", word):                                     # P1Y (a duration in XBRL)
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", word):                              # 2026-03-28
        return True
    if re.fullmatch(r"\d{10}", word):                                        # 0000320193 (company number)
        return True
    return False


def junk_share(text):
    # share of the words that are machine data (0.0 = clean text)
    all_words = text.split()
    if len(all_words) == 0:
        return 1.0
    junk_words = [w for w in all_words if is_xbrl_word(w)]
    return len(junk_words) / len(all_words)


def remove_xbrl_junk(text):
    # drop every line where more than 20% of the words are machine data
    clean_lines = []
    for line in text.split("\n"):
        if line.strip() == "" or junk_share(line) <= 0.2:
            clean_lines.append(line)
    text = "\n".join(clean_lines)
    return re.sub(r"\n\s*\n\s*", "\n\n", text).strip()


def download_from_sec_website(cik, accession):
    """Find the main 10-Q document on the filing's index page and download it."""
    folder = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}"
    index_page = requests.get(f"{folder}/{accession}-index.htm", headers=SEC_HEADERS, timeout=60)
    time.sleep(0.2)
    if index_page.status_code != 200:
        return None

    # the index page has a table: Seq | Description | Document | Type | Size
    soup = BeautifulSoup(index_page.text, "html.parser")
    for table_row in soup.find_all("tr"):
        cells = table_row.find_all("td")
        if len(cells) < 4 or cells[3].get_text(strip=True) != "10-Q":
            continue
        link = cells[2].find("a")
        if link is None:
            continue
        url = "https://www.sec.gov" + link["href"].replace("/ix?doc=", "")
        document = requests.get(url, headers=SEC_HEADERS, timeout=60)
        time.sleep(0.2)
        if document.status_code == 200:
            return document.text
    return None


def download_filing_html(cik, accession):
    """First try the SEC website directly, then the edgartools library."""
    try:
        html = download_from_sec_website(cik, accession)
        if html:
            return html
    except Exception as error:
        print("   SEC website failed:", str(error)[:100])
    try:
        from edgar import find
        return find(accession).html()
    except Exception as error:
        print("   edgartools failed:", str(error)[:100])
    return None


sample = quarters.sort_values("period").groupby(["ticker", "year"]).head(1)
sample = sample.sort_values(["ticker", "period"]).reset_index(drop=True)
print(len(sample), "filings in the sample")

filing_texts = {}     # accession number -> clean text of the filing

for _, row in sample.iterrows():
    html = download_filing_html(row["cik"], row["accession"])
    if html is None:
        print(f"{row['ticker']} {row['period']}  could not find the 10-Q document")
        continue
    text = remove_xbrl_junk(html_to_text(html))
    filing_texts[row["accession"]] = text
    print(f"{row['ticker']} {row['period']}  {len(text):>9,} characters   machine data: {junk_share(text):.1%}")

missing = set(sample.accession) - set(filing_texts)
print("Missing filings:", missing if missing else "none")
sample = sample[sample.accession.isin(filing_texts)].reset_index(drop=True)

36 filings in the sample
AAPL 2015-03-28    196,471 characters   machine data: 0.0%
AAPL 2016-03-26    176,056 characters   machine data: 0.0%
AAPL 2017-04-01    177,983 characters   machine data: 0.0%
AAPL 2018-03-31    186,139 characters   machine data: 0.0%
AAPL 2019-03-30    187,082 characters   machine data: 0.0%
AAPL 2020-03-28    126,333 characters   machine data: 0.0%
AAPL 2021-03-27    111,253 characters   machine data: 0.0%
AAPL 2022-03-26     67,941 characters   machine data: 0.0%
AAPL 2023-04-01     64,597 characters   machine data: 0.0%
AAPL 2024-03-30     63,114 characters   machine data: 0.0%
AAPL 2025-03-29     75,899 characters   machine data: 0.0%
AAPL 2026-03-28     89,504 characters   machine data: 0.0%
MSFT 2015-03-31    224,821 characters   machine data: 0.0%
MSFT 2016-03-31    212,036 characters   machine data: 0.0%
MSFT 2017-03-31    213,821 characters   machine data: 0.0%
MSFT 2018-03-31    233,737 characters   machine data: 0.0%
MSFT 2019-03-31    207,109 char

## 6. Function to ask the model a question
The model must answer in JSON, with the amounts **in millions of USD**.
If the model breaks down (loops, bad JSON), we record it as a failure and keep going.

In [ ]:
import json
import ollama

from transformers import AutoTokenizer

client = ollama.Client(host="http://localhost:11434")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)


def count_tokens(text):
    return len(tokenizer.encode(text))

INSTRUCTIONS = (
    "You are a financial analyst. Answer only with JSON. "
    "Give every amount in MILLIONS of US dollars (for example $1.2 billion -> 1200). "
    "If you do not know a figure, write null. Never guess."
)

ANSWER_FORMAT = {
    "type": "object",
    "properties": {
        "revenue_millions": {"anyOf": [{"type": "number"}, {"type": "null"}]},
        "net_income_millions": {"anyOf": [{"type": "number"}, {"type": "null"}]},
        "evidence": {"type": "string"},
    },
    "required": ["revenue_millions", "net_income_millions", "evidence"],
}


def make_question(row):
    return (f"What were the total revenue and the net income of {row['company']} ({row['ticker']}) "
            f"for the three months (fiscal quarter) ended {row['period']}? "
            "Use the quarterly figures, not year-to-date. "
            "In 'evidence', quote ONE short line (max 30 words) where you found them.")


def ask_model(model_name, question, context, context_size):
    if context is None:
        prompt = question
    else:
        prompt = "TEXT FROM THE FILING:\n" + context + "\n\nQUESTION:\n" + question

    result = {"revenue_millions": None, "net_income_millions": None, "evidence": None,
              "prompt_tokens": None, "seconds": None, "error": None}

    # Safety check: if the prompt is bigger than the context window, Ollama silently
    # throws away the beginning of the text. We refuse instead, so we never score a cut prompt.
    prompt_size = count_tokens(INSTRUCTIONS + prompt) + 100
    if prompt_size > context_size - 512:
        result["error"] = f"prompt too long: {prompt_size} tokens for a {context_size} window"
        return result

    start_time = time.time()
    try:
        response = client.chat(
            model=model_name,
            messages=[{"role": "system", "content": INSTRUCTIONS},
                      {"role": "user", "content": prompt}],
            format=ANSWER_FORMAT,
            think=False if model_name.startswith("qwen3") else None,
            options={"temperature": 0, "num_ctx": context_size,
                     "num_predict": 512, "repeat_penalty": 1.1},
        )
        answer = json.loads(response.message.content)
        result["revenue_millions"] = answer.get("revenue_millions")
        result["net_income_millions"] = answer.get("net_income_millions")
        result["evidence"] = answer.get("evidence")
        result["prompt_tokens"] = response.prompt_eval_count
    except Exception as error:
        result["error"] = str(error)[:200]
    result["seconds"] = round(time.time() - start_time, 1)
    return result


# quick test
print(ask_model(MODEL, "Apple had $1.5 billion revenue and $300 million net income. Report them.", None, 2048))

## 7. RAG: cut each filing into passages and build the search
1. **Cut** the filing into passages of ~1,800 characters. A table is never cut in half.
2. **Embed** every passage (turn it into a vector of numbers that represents its meaning).
3. **Search** with two methods and combine them:
   - meaning search (vectors) finds passages that talk about the same thing,
   - keyword search (BM25) finds exact words like "Total net sales".

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi


def split_into_passages(text, max_size=1800):
    paragraphs = re.split(r"\n\s*\n", text)
    passages = []
    current = ""
    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        is_table = paragraph.startswith("|")
        if is_table:
            # keep the end of the text before the table (its title, "In millions", ...)
            title = current[-300:]
            if current:
                passages.append(current)
            for start in range(0, len(paragraph), 6000):     # very long tables: cut every 6000 characters
                passages.append(title + "\n\n" + paragraph[start:start + 6000])
            current = ""
        elif len(paragraph) > max_size:
            # a very long paragraph: cut it in pieces of max_size characters
            if current:
                passages.append(current)
            for start in range(0, len(paragraph), max_size):
                passages.append(paragraph[start:start + max_size])
            current = ""
        elif len(current) + len(paragraph) > max_size:
            passages.append(current)
            current = paragraph
        else:
            current = current + "\n\n" + paragraph
    if current:
        passages.append(current)
    return [p.strip() for p in passages if p.strip()]


def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def embed(texts, prefix):
    vectors = []
    for i in range(0, len(texts), 32):
        batch = [prefix + t for t in texts[i:i + 32]]
        vectors.extend(client.embed(model=EMBED_MODEL, input=batch).embeddings)
    vectors = np.array(vectors)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)   # length 1 -> dot product = similarity


search_index = {}    # accession -> passages, vectors, keyword index

for accession, text in filing_texts.items():
    passages = split_into_passages(text)
    search_index[accession] = {
        "passages": passages,
        "vectors": embed(passages, "search_document: "),
        "keywords": BM25Okapi([words(p) for p in passages]),
    }
    print(accession, len(passages), "passages")


# Check the data BEFORE running the experiment: is the real revenue in the text, and how far in?
# A filing where the revenue cannot be found in the text was not converted correctly:
# we remove it from the sample, so that bad data never counts as a mistake of RAG or long context.
print("\nticker  period      tokens  revenue in text?  revenue found at token")
bad_filings = []
for _, row in sample.iterrows():
    text = filing_texts[row["accession"]]
    revenue = f"{round(row['true_revenue'] / 1e6):,}"
    position = text.find(revenue)
    if position >= 0:
        found_at = count_tokens(text[:position])
    else:
        found_at = None
        bad_filings.append(row["accession"])
    print(f"{row['ticker']:6}  {row['period']}  {count_tokens(text):>7,}  {str(position >= 0):16}  {found_at}")

print(f"\n{len(bad_filings)} filing(s) removed from the sample because the revenue is not in the text")
sample = sample[~sample.accession.isin(bad_filings)].reset_index(drop=True)
print(len(sample), "filings kept for RAG and long context")
if len(sample) < 10:
    raise RuntimeError("Too few clean filings: the download or the cleaning did not work. Look at the table above.")


def search(accession, query, top_k=RAG_TOP_K):
    index = search_index[accession]
    query_vector = embed([query], "search_query: ")[0]

    meaning_scores = index["vectors"] @ query_vector
    keyword_scores = index["keywords"].get_scores(words(query))

    meaning_ranking = np.argsort(-meaning_scores)[:30]
    keyword_ranking = np.argsort(-keyword_scores)[:30]

    # Reciprocal Rank Fusion: a passage ranked high by either method gets a high score
    combined = {}
    for rank, passage_id in enumerate(meaning_ranking):
        combined[passage_id] = combined.get(passage_id, 0) + 1 / (60 + rank)
    for rank, passage_id in enumerate(keyword_ranking):
        combined[passage_id] = combined.get(passage_id, 0) + 1 / (60 + rank)

    # take the best passages until we have top_k of them or the context window is full
    budget = RAG_CONTEXT - 1500
    best = []
    used_tokens = 0
    for passage_id in sorted(combined, key=combined.get, reverse=True):
        size = count_tokens(index["passages"][passage_id])
        if used_tokens + size > budget:
            continue
        best.append(passage_id)
        used_tokens += size
        if len(best) == top_k:
            break

    best.sort()   # keep the order of the document
    return "\n\n---\n\n".join(index["passages"][i] for i in best)


def cut_to_fit(text, context_size):
    """Long context mode: keep only the beginning of the filing that fits in the context window."""
    tokens = tokenizer.encode(text)
    max_tokens = context_size - 1500
    if len(tokens) <= max_tokens:
        return text, False
    return tokenizer.decode(tokens[:max_tokens]), True


def number_is_in_text(true_value, text):
    """Did the model receive the right number? (e.g. 109,417 for $109.4 billion)"""
    if text is None or true_value is None or pd.isna(true_value):
        return None
    clean_text = text.replace("$", "").replace(" ", "")
    in_millions = f"{round(true_value / 1e6):,}"
    in_thousands = f"{round(true_value / 1e3):,}"
    return in_millions in clean_text or in_thousands in clean_text

0001193125-15-153166 229 passages
0001193125-16-559625 203 passages
0001628280-17-004790 209 passages
0000320193-18-000070 213 passages
0000320193-19-000066 212 passages
0000320193-20-000052 168 passages
0000320193-21-000056 120 passages
0000320193-22-000059 85 passages
0000320193-23-000064 84 passages
0000320193-24-000069 77 passages
0000320193-25-000057 87 passages
0000320193-26-000013 98 passages
0001193125-15-144151 283 passages
0001193125-16-550254 287 passages
0001564590-17-007547 321 passages
0001564590-18-009307 330 passages
0001564590-19-012709 313 passages
0001564590-20-019706 326 passages
0001564590-21-020891 338 passages
0001564590-22-015675 348 passages
0000950170-23-014423 230 passages
0000950170-24-048288 234 passages
0000950170-25-061046 234 passages
0001193125-26-191507 223 passages
0001045810-15-000097 123 passages
0001045810-16-000275 141 passages
0001045810-17-000075 128 passages
0001045810-18-000080 118 passages
0001045810-19-000079 135 passages
0001045810-20-00006

## 8. Run the experiment
- A. No RAG: every quarter, for each model in `NO_RAG_MODELS`
- B. RAG and C. Long context: the sample only

Results are saved to `results.csv` after every answer. If Colab disconnects, re-run everything:
questions already answered (in `results.csv`) are skipped.

In [ ]:
RAG_QUERY = "condensed consolidated statements of operations total net sales revenue net income three months ended {period}"

# list every question to ask: (mode, model, row of the quarter)
jobs = []
for model_name in NO_RAG_MODELS:
    for _, row in quarters.iterrows():
        jobs.append((f"A. No RAG ({model_name})", model_name, row))
for _, row in sample.iterrows():
    jobs.append(("B. RAG", MODEL, row))
for _, row in sample.iterrows():
    jobs.append(("C. Long context", MODEL, row))

# load what was already done
if START_FRESH and os.path.exists("results.csv"):
    os.remove("results.csv")
if os.path.exists("results.csv"):
    results_list = pd.read_csv("results.csv").to_dict("records")
else:
    results_list = []
already_done = set((r["mode"], r["ticker"], r["period"]) for r in results_list)
print(len(jobs), "questions in total,", len(already_done), "already done")

for mode, model_name, row in jobs:
    if (mode, row["ticker"], row["period"]) in already_done:
        continue

    truncated = False
    if mode.startswith("A."):
        context, context_size = None, NO_RAG_CONTEXT
    elif mode.startswith("B."):
        context = search(row["accession"], RAG_QUERY.format(period=row["period"]))
        context_size = RAG_CONTEXT
    else:
        context, truncated = cut_to_fit(filing_texts[row["accession"]], LONG_CONTEXT)
        context_size = LONG_CONTEXT

    answer = ask_model(model_name, make_question(row), context, context_size)

    results_list.append({
        "mode": mode, "model": model_name,
        "ticker": row["ticker"], "period": row["period"], "year": row["year"],
        "in_sample": row["accession"] in filing_texts,
        "true_revenue": row["true_revenue"], "true_net_income": row["true_net_income"],
        "revenue_millions": answer["revenue_millions"],
        "net_income_millions": answer["net_income_millions"],
        "evidence": answer["evidence"],
        "seconds": answer["seconds"], "prompt_tokens": answer["prompt_tokens"],
        "truncated": truncated,
        "answer_was_in_context": number_is_in_text(row["true_revenue"], context),
        "error": answer["error"],
    })
    pd.DataFrame(results_list).to_csv("results.csv", index=False)

    status = "ERROR" if answer["error"] else ""
    print(f"{mode:28} {row['ticker']} {row['period']}  {answer['seconds']:>5}s  {status}")

results = pd.DataFrame(results_list)

## 9. Score every answer against the ground truth
| Grade | Meaning |
|---|---|
| exact | within 0.5% of the real value |
| close | within 10% (the model "roughly remembers") |
| wrong | a number, but a wrong one (hallucination) |
| not found | the model said it does not know |
| model failure | the model broke down (loop, bad JSON) |

In [ ]:
def grade(predicted_millions, true_value, error):
    if true_value is None or pd.isna(true_value):
        return None
    if isinstance(error, str) and error:
        return "model failure"
    if predicted_millions is None or pd.isna(predicted_millions):
        return "not found"
    true_millions = true_value / 1e6
    difference = abs(predicted_millions - true_millions) / abs(true_millions)
    if difference <= EXACT_TOLERANCE:
        return "exact"
    if difference <= CLOSE_TOLERANCE:
        return "close"
    return "wrong"


results["revenue_grade"] = [grade(p, t, e) for p, t, e in
                            zip(results.revenue_millions, results.true_revenue, results.error)]
results["net_income_grade"] = [grade(p, t, e) for p, t, e in
                               zip(results.net_income_millions, results.true_net_income, results.error)]

# one line per graded number (revenue and net income together)
revenue_part = results[["mode", "ticker", "period", "year", "in_sample", "revenue_grade"]].rename(
    columns={"revenue_grade": "grade"})
income_part = results[["mode", "ticker", "period", "year", "in_sample", "net_income_grade"]].rename(
    columns={"net_income_grade": "grade"})
grades = pd.concat([revenue_part.assign(metric="revenue"), income_part.assign(metric="net income")],
                   ignore_index=True)
grades = grades.dropna(subset=["grade"])

# Fair comparison: all modes on the SAME sample quarters
sample_grades = grades[grades.in_sample]
summary = pd.crosstab(sample_grades["mode"], sample_grades["grade"], normalize="index").mul(100).round(0)
summary["avg seconds"] = results[results.in_sample].groupby("mode").seconds.mean().round(1)
summary["avg prompt tokens"] = results[results.in_sample].groupby("mode").prompt_tokens.mean().round(0)
summary["right number was in the text (%)"] = (
    results[results.in_sample].groupby("mode").answer_was_in_context.mean().mul(100).round(0))
summary.to_csv("summary.csv")
summary

## 10. Charts for the presentation

In [ ]:
import plotly.express as px

GRADE_COLORS = {"exact": "#2e7d32", "close": "#81c784", "not found": "#9e9e9e",
                "wrong": "#c62828", "model failure": "#6a1b9a"}
GRADE_ORDER = ["exact", "close", "wrong", "not found", "model failure"]

# Chart 1: % of good answers (exact or close) per year, for each mode
grades["good"] = grades["grade"].isin(["exact", "close"])
by_year = grades.groupby(["mode", "year"]).good.mean().mul(100).reset_index()
chart1 = px.line(by_year, x="year", y="good", color="mode", markers=True,
                 title="Share of good answers (exact or within 10%) by year",
                 labels={"good": "% good answers", "year": "Year"})
chart1.update_yaxes(range=[0, 105])
chart1.show()

# Chart 2: what the No RAG model did each year
no_rag = grades[grades["mode"].str.startswith("A.")]
chart2 = px.histogram(no_rag, x="year", color="grade", barnorm="percent",
                      category_orders={"grade": GRADE_ORDER}, color_discrete_map=GRADE_COLORS,
                      facet_row="mode" if no_rag["mode"].nunique() > 1 else None,
                      title="No RAG: answers by year")
chart2.update_layout(yaxis_title="% of answers", bargap=0.1)
chart2.show()

# Chart 3: real revenue vs what the No RAG model said
revenue_lines = []
for _, row in quarters.iterrows():
    revenue_lines.append({"ticker": row["ticker"], "period": row["period"], "source": "Real (SEC)",
                          "revenue_bn": row["true_revenue"] / 1e9})
for _, row in results[results["mode"].str.startswith("A.")].iterrows():
    if pd.notna(row["revenue_millions"]):
        revenue_lines.append({"ticker": row["ticker"], "period": row["period"], "source": row["mode"],
                              "revenue_bn": row["revenue_millions"] / 1e3})
chart3 = px.line(pd.DataFrame(revenue_lines).sort_values("period"), x="period", y="revenue_bn",
                 color="source", facet_col="ticker", markers=True,
                 title="Quarterly revenue: real vs answered without RAG",
                 labels={"revenue_bn": "Revenue (USD bn)", "period": "Quarter end"})
chart3.update_yaxes(matches=None, showticklabels=True)
chart3.show()

# Chart 4: outcomes by mode on the same sample
chart4 = px.histogram(sample_grades, x="mode", color="grade", barnorm="percent",
                      category_orders={"grade": GRADE_ORDER}, color_discrete_map=GRADE_COLORS,
                      title="Answers by mode (same quarters for every mode)")
chart4.update_layout(yaxis_title="% of answers")
chart4.show()

# Chart 5: cost of each mode
cost = results[results.in_sample].groupby("mode")[["seconds", "prompt_tokens"]].mean().reset_index()
chart5 = px.bar(cost.melt(id_vars="mode"), x="mode", y="value", color="mode", facet_col="variable",
                title="Average time (s) and prompt size (tokens) per question")
chart5.update_yaxes(matches=None, showticklabels=True)
chart5.show()

for number, chart in enumerate([chart1, chart2, chart3, chart4, chart5], start=1):
    chart.write_html(f"chart_{number}.html")

## 11. Look at one example in detail (for a "why did it fail?" slide)
Change `EXAMPLE` to look at another row of the sample.

In [ ]:
EXAMPLE = 0
row = sample.iloc[EXAMPLE]

print("QUESTION:", make_question(row))
print(f"\nREAL: revenue = {row['true_revenue'] / 1e6:,.0f} M$, net income = {row['true_net_income'] / 1e6:,.0f} M$\n")

answers = results[(results.ticker == row["ticker"]) & (results.period == row["period"])]
print(answers[["mode", "revenue_millions", "net_income_millions", "revenue_grade", "evidence"]].to_string())

print("\n===== PASSAGES FOUND BY THE RAG SEARCH =====\n")
print(search(row["accession"], RAG_QUERY.format(period=row["period"]))[:4000])

---
# Part 2: questions a trader would really ask

In Part 1, the answers (revenue, net income) already exist in a database, so AI is not really needed.
Here we ask questions whose answer is **only in the text** of the filing.

We use the **most recent filing of each company in the sample**. It is after the model's training cutoff,
so without RAG the model cannot know the answer.

There is no automatic "right answer" here: we read the answers side by side and judge them.
To help, we check one thing automatically: **are the numbers quoted in the answer really in the filing?**
A number that is not in the filing is a strong sign of hallucination.

## 12. The questions

In [ ]:
TRADER_QUESTIONS = [
    "What does management say about demand and the outlook for the coming quarters?",
    "What were the main reasons revenue changed compared with the same quarter last year?",
    "Why did the gross margin change compared with the same quarter last year?",
    "Which risks or uncertainties does the company highlight in this report?",
    "How much cash did the company return to shareholders (share buybacks and dividends) during the quarter?",
    "Does the company mention a customer that represents a large share of its revenue? If yes, what share?",
]

# the most recent filing of each company in the sample
latest_filings = sample.sort_values("period").groupby("ticker").tail(1).reset_index(drop=True)
latest_filings[["ticker", "company", "period"]]

## 13. Ask every question with the 3 modes
Same model and same rules as Part 1. The answer is free text this time (no JSON).
Results are saved to `trader_answers.csv` after every answer, and already answered questions are skipped.

In [ ]:
TEXT_INSTRUCTIONS = (
    "You are a financial analyst helping a trader. Answer in 3 to 5 sentences. "
    "Be precise and give numbers when you can. "
    "If you were given text from a filing, use only that text and quote it when possible. "
    "If you do not know the answer, say 'I don't know'. Never invent facts."
)


def ask_model_free_text(model_name, question, context, context_size):
    if context is None:
        prompt = question
    else:
        prompt = "TEXT FROM THE FILING:\n" + context + "\n\nQUESTION:\n" + question

    result = {"answer": None, "prompt_tokens": None, "seconds": None, "error": None}

    prompt_size = count_tokens(TEXT_INSTRUCTIONS + prompt) + 100
    if prompt_size > context_size - 512:
        result["error"] = f"prompt too long: {prompt_size} tokens for a {context_size} window"
        return result

    start_time = time.time()
    try:
        response = client.chat(
            model=model_name,
            messages=[{"role": "system", "content": TEXT_INSTRUCTIONS},
                      {"role": "user", "content": prompt}],
            think=False if model_name.startswith("qwen3") else None,
            options={"temperature": 0, "num_ctx": context_size,
                     "num_predict": 400, "repeat_penalty": 1.1},
        )
        result["answer"] = response.message.content.strip()
        result["prompt_tokens"] = response.prompt_eval_count
    except Exception as error:
        result["error"] = str(error)[:200]
    result["seconds"] = round(time.time() - start_time, 1)
    return result


def numbers_in(text):
    # numbers like 1,234 or 45.6 or 15 (we skip years like 2025 and single digits)
    found = re.findall(r"\d[\d,]*\.?\d*", text or "")
    numbers = []
    for number in found:
        number = number.strip(",.")
        is_year = len(number) == 4 and number.isdigit() and 1990 <= int(number) <= 2035
        if len(number) >= 2 and not is_year:
            numbers.append(number)
    return numbers


def check_numbers(answer, filing_text):
    # how many numbers of the answer can be found in the filing?
    numbers = numbers_in(answer)
    if len(numbers) == 0:
        return "no numbers"
    clean_filing = filing_text.replace(" ", "")
    found = 0
    for number in numbers:
        if number in clean_filing:
            found += 1
    return f"{found}/{len(numbers)} found in filing"


if START_FRESH and os.path.exists("trader_answers.csv"):
    os.remove("trader_answers.csv")
if os.path.exists("trader_answers.csv"):
    trader_list = pd.read_csv("trader_answers.csv").to_dict("records")
else:
    trader_list = []
already_done = set((r["mode"], r["ticker"], r["question"]) for r in trader_list)

MODE_SETTINGS = [
    ("A. No RAG", NO_RAG_CONTEXT),
    ("B. RAG", RAG_CONTEXT),
    ("C. Long context", LONG_CONTEXT),
]

for mode, context_size in MODE_SETTINGS:
    for _, row in latest_filings.iterrows():
        filing_text = filing_texts[row["accession"]]
        for question in TRADER_QUESTIONS:
            if (mode, row["ticker"], question) in already_done:
                continue

            full_question = (f"About {row['company']} ({row['ticker']}), for the fiscal quarter "
                             f"ended {row['period']}: {question}")
            truncated = False
            if mode == "A. No RAG":
                context = None
            elif mode == "B. RAG":
                context = search(row["accession"], question)
            else:
                context, truncated = cut_to_fit(filing_text, context_size)

            answer = ask_model_free_text(MODEL, full_question, context, context_size)

            trader_list.append({
                "mode": mode, "ticker": row["ticker"], "period": row["period"],
                "question": question,
                "answer": answer["answer"],
                "numbers_check": check_numbers(answer["answer"], filing_text) if answer["answer"] else None,
                "seconds": answer["seconds"], "prompt_tokens": answer["prompt_tokens"],
                "truncated": truncated, "error": answer["error"],
            })
            pd.DataFrame(trader_list).to_csv("trader_answers.csv", index=False)
            print(f"{mode:16} {row['ticker']}  {answer['seconds']:>5}s  {question[:50]}")

trader_answers = pd.DataFrame(trader_list)

## 14. Read the answers side by side
- The printout below is easy to read in Colab.
- `trader_answers.html` is a table you can open in a browser and screenshot for the slides.
- To grade the answers, open `trader_answers.csv` in Excel and add a column with your grade (good / partial / wrong).

In [ ]:
for _, filing_row in latest_filings.iterrows():
    for question in TRADER_QUESTIONS:
        print("=" * 100)
        print(f"{filing_row['ticker']} ({filing_row['period']})  {question}")
        print("=" * 100)
        answers = trader_answers[(trader_answers.ticker == filing_row["ticker"]) &
                                 (trader_answers.question == question)]
        for _, answer_row in answers.iterrows():
            text = answer_row["answer"] if isinstance(answer_row["answer"], str) else f"ERROR: {answer_row['error']}"
            print(f"\n--- {answer_row['mode']}   [numbers: {answer_row['numbers_check']}]")
            print(text)
        print()

# one row per question, one column per mode
side_by_side = trader_answers.pivot_table(index=["ticker", "question"], columns="mode",
                                          values="answer", aggfunc="first")
side_by_side.to_html("trader_answers.html")

# quick summary of the automatic number check
print(trader_answers.groupby("mode").numbers_check.value_counts())

## 15. Download the results

In [ ]:
from google.colab import files

for file_name in ["results.csv", "summary.csv", "chart_1.html", "chart_2.html",
                  "chart_3.html", "chart_4.html", "chart_5.html",
                  "trader_answers.csv", "trader_answers.html"]:
    files.download(file_name)